# V037: conditional final all-approved capacity fit

Default inert. Requires completed, independently audited cold-F success with explicit hashes; stage-only or A-only evidence cannot start this fit. Reuses five accepted all-data checkpoints/caches after strict audits; fits capacity190 on all67905. No automatic submission.

In [ ]:
import os,json,hashlib,gc,time,fcntl,copy
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import torch
V037_SOURCE_DEPENDENCIES={"controlled_experiments_v013.ipynb":"56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a","final_fit_v011.ipynb":"35abc3371ddaf717ca3fc7ff908f4e06517e642c0d0fc6b089342870f533c6de","research/postprocessing_components_v016.ipynb":"4ee67389039846557d74cc1c55884c9a99f0e571745af4febefa4b5c3bce6a0b"}
V037_ACCEPTED_MANIFEST_SHA="3746753aa6cd7eaa306f2279e24d5a5231b6bae9c700bcd703e54b756a823aaf"
V037_ACCEPTED_CSV_SHA="c344a485f5ef1290aadf5dfd379333009efab674b7ccdff6f361e225a4b8b77a"
V037_NOTEBOOK="final_fit_v037.ipynb"
V037_LAPLACE_SHA="b6bd59bced1fad03882f27d9faec51deec93b3ab10afa9f06b8cef36c4d5260d"
V037_BEST_OFFICIAL_CSV_SHA="274668c4dc14e9d20a554c738859fca37e448836797d7260ec7b11464dcde3e2"

def v037_require(c,m):
    if not bool(c):raise ValueError(m)
def v037_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def v037_code(p):return "\n".join("".join(c["source"]) for c in json.loads(Path(p).read_text())["cells"] if c["cell_type"]=="code")
def v037_path(root,value):
    marker="/ventilator-pressure-prediction/";value=str(value);p=(root/(value.split(marker,1)[1] if marker in value else value)).resolve()
    v037_require(p.is_relative_to(root),"Path outside project");return p

def v037_confirmation_gate(root,expected_F_result_sha,expected_F_audit_sha):
    """No F labels read. Require a completed cold-F decision and bound full audit."""
    root=Path(root);E=root/'outputs/confirmation_v036';paths=[E/'confirmation_result.json',root/'research/confirmation_review_v036.json'];expected=[expected_F_result_sha,expected_F_audit_sha]
    for p,h in zip(paths,expected):
        v037_require(isinstance(h,str) and len(h)==64 and all(c in '0123456789abcdef' for c in h),'Missing explicit reviewed F result/audit digest')
        v037_require(p.is_file() and v037_sha(p)==h,'Reviewed F prerequisite changed: '+str(p))
    result,audit=[json.loads(p.read_text()) for p in paths];v037_require(result['status']=='complete' and result['passed'] and audit['status']=='passed' and audit['passed'],'Complete independently audited F pass required; partial stage cannot authorize final fit')
    v037_require(audit['confirmation_result_sha256']==expected_F_result_sha and audit['selection_sha256']==result['selection_sha256'],'F result and independent audit differ')
    pp=root/'research/confirmation_protocol_v036.json';sp=root/'research/confirmation_selection_v036.json';v037_require(v037_sha(pp)=='e9fb1fc5fd7a342fd744f8ed2d7bcdbb4eab3d3e5be8f12da6f93575374ab0f4' and v037_sha(sp)=='ec2feebb872eaaf963ca27a1c1f61c23af0f6f1aac24b3d4a39986100b8f54ec','Frozen F scientific choice/protocol changed')
    plan=json.loads(pp.read_text());choice=json.loads(sp.read_text());lock=json.loads((E/'locked_before_F.json').read_text());canonical=hashlib.sha256(json.dumps(lock,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
    v037_require(result['source_code_sha256']=='27f89eb9f1a749e032040082b70513fc9990311eee87fcddbf7e6d5806f8138b' and result['protocol_sha256']==v037_sha(pp) and result['selection_sha256']==canonical and result['locked_file_sha256']==v037_sha(E/'locked_before_F.json'),'F source/runtime-lock identity changed')
    v037_require(result['winner_name']==plan['winner_name']==choice['winner']['name'] and result['winner_weights']==plan['winner_weights']==choice['winner_weights'] and result['baseline_weights']==plan['baseline_weights']==choice['baseline_weights'],'Only frozen lowest eligible A recipe may advance')
    v037_require(set(result['records'])==set(plan['members']) and len(result['records'])==7 and lock['members']==plan['members'] and result['cold_fit_all_members'] and result['no_F_epoch_or_weight_selection'] and result['original_A_validation_excluded'] and result['confirmation_is_resampling_evidence'] and result['no_runner_up_after_failure'],'Seven cold F members and no post-F search required')
    v037_require(result['gain']>0 and result['interval']['high']<0 and audit['gain']==result['gain'] and audit['baseline_MAE']==result['metrics']['baseline']['mae'] and audit['candidate_MAE']==result['metrics']['candidate']['mae'],'F native MAE/whole-breath acceptance failed')
    for name,record in result['records'].items():
        v037_require(record['identity']['selection_sha256']==canonical and record['identity']['recipe']==plan['members'][name],'F member recipe/selection changed')
        for key,hkey in [('checkpoint_file','checkpoint_sha256'),('last_checkpoint_file','last_checkpoint_sha256'),('report_file','report_sha256')]:v037_require(v037_sha(v037_path(root,record[key]))==record[hkey],'Verified cold F artifact changed: '+name)
    ids=np.load(E/'F_row_ids.npy',allow_pickle=False);v037_require(ids.dtype==np.int64 and hashlib.sha256(np.ascontiguousarray(ids).tobytes()).hexdigest()==result['F_row_ids_array_sha256'],'Verified F row IDs changed');predictions={}
    for name,a in result['prediction_artifacts'].items():
        path=v037_path(root,a['file']);v037_require(v037_sha(path)==a['file_sha256'],'F native prediction cache changed');p=np.load(path,allow_pickle=False);v037_require(p.dtype==np.float64 and p.shape==ids.shape and np.isfinite(p).all() and hashlib.sha256(np.ascontiguousarray(p).tobytes()).hexdigest()==a['array_sha256'],'Invalid verified F native prediction cache');predictions[name]=p
    grid=np.load(E/'F_fit_pressure_codebook.npy',allow_pickle=False)
    # Pure frozen policy reconstruction does not need validation labels.
    policy=v037_code(root/'research/postprocessing_components_v016.ipynb');v037_require(hashlib.sha256(policy.encode()).hexdigest()=='4ee67389039846557d74cc1c55884c9a99f0e571745af4febefa4b5c3bce6a0b','Frozen pressure-grid policy changed');ns={};exec(compile(policy,'postprocessing_components_v016.ipynb','exec'),ns)
    for name,weights in [('baseline',plan['baseline_weights']),('candidate',plan['winner_weights'])]:
        predicted=ns['selected_postprocessing_v016']({k:predictions[k] for k in weights},weights,grid);v037_require(np.array_equal(predicted,np.load(E/(name+'_F_predictions.npy'))),'Verified F postprocessing arrays differ')
    prediction_lock=json.loads((E/'predictions_locked_before_F_pressure.json').read_text());v037_require(prediction_lock['selection_sha256']==canonical and prediction_lock['no_validation_pressure_loaded'] and prediction_lock['baseline_sha256']==v037_sha(E/'baseline_F_predictions.npy') and prediction_lock['candidate_sha256']==v037_sha(E/'candidate_F_predictions.npy') and prediction_lock['row_ids_sha256']==v037_sha(E/'F_row_ids.npy') and prediction_lock['grid_sha256']==v037_sha(E/'F_fit_pressure_codebook.npy'),'F predictions not bound to prelabel lock')
    return plan,dict(F_result_sha256=expected_F_result_sha,F_audit_sha256=expected_F_audit_sha,selection_sha256=canonical,scientific_A_selection_sha256=v037_sha(sp),F_protocol_sha256=v037_sha(pp))

def v037_laplace_factory(spec,n_features,preprocess):return v029_laplace_factory(spec,n_features,preprocess)
v037_laplace_factory.__module__='final_fit_v037'

def v037_fit_capacity(root,out,recipe,data,gate,sources,resume):
    directory=out/'capacity_laplace_curve';directory.mkdir(exist_ok=True);checkpoint=directory/'model.pt';rp=directory/'record.json';report_path=directory/'report.json'
    identity=dict(member_id='capacity_laplace_curve',recipe=recipe,gate=gate,sources=sources,training_group_sha256=data['group_sha256'],training_data_fingerprints=data['data_hashes']);v011_immutable_json(directory/'identity.json',identity,resume)
    if rp.exists():
        record=json.loads(rp.read_text());v037_require(resume and record['identity']==identity,'Existing all-data capacity identity differs')
        for key,hkey in [('checkpoint_path','checkpoint_sha256'),('last_checkpoint_path','last_checkpoint_sha256'),('report_file','report_sha256')]:v037_require(v037_sha(v037_path(root,record[key]))==record[hkey],'Existing final capacity artifact changed')
        return record
    if report_path.exists():
        v037_require(resume,'Completed orphan report requires explicit resume');report=json.loads(report_path.read_text());v037_require(report['checkpoint_sha256']==v037_sha(checkpoint),'Orphan all-data report/model differ')
    else:
        report=fit_fixed_v029(data['x'],data['y'],data['out'],recipe['model_spec'],recipe['train_cfg'],recipe['seed'],recipe['lr_schedule'],checkpoint,epoch_log_path=directory/'epochs.jsonl',feature_names=data['names'],device='cuda',resume=resume and _last_path(checkpoint).exists(),model_factory=v037_laplace_factory,loss_kind='laplace_curve',arm='v037_all_capacity_laplace_curve',source_fingerprints=sources)
        v011_write_json(report_path,report)
    v037_require(report['completed'] and not report['validation_used'] and report['training_breaths']==67905 and report['executed_epochs']==recipe['selected_epochs']==190 and report['executed_lr_schedule']==recipe['lr_schedule'] and report['actual_training_objective']=='laplace_curve','All-data capacity fixed budget/objective incomplete')
    model,prep,payload=load_checkpoint_v029(checkpoint,device='cpu',model_factory=v037_laplace_factory,expected_arm='v037_all_capacity_laplace_curve',expected_loss_kind='laplace_curve',source_fingerprints=sources)
    expected=fit_preprocessor(data['x'],data['y'],data['out'],recipe['model_spec'],data['names'],scaler_phase='all',residual_baseline=False);v037_require(_v8_statistics_sha256(prep)==_v8_statistics_sha256(expected),'Fresh all-training capacity statistics differ');del model,prep,payload,expected
    record=dict(identity=identity,member_id='capacity_laplace_curve',recipe=recipe,checkpoint_path=str(checkpoint.relative_to(root)),checkpoint_sha256=v037_sha(checkpoint),last_checkpoint_path=str(_last_path(checkpoint).relative_to(root)),last_checkpoint_sha256=v037_sha(_last_path(checkpoint)),report_file=str(report_path.relative_to(root)),report_sha256=v037_sha(report_path),arm='v037_all_capacity_laplace_curve',training_breaths=67905,selected_epochs=190,source_fingerprints=sources)
    v011_write_json(rp,record);return record

def v037_infer_capacity(root,out,record,sources,resume):
    known=pd.read_csv(root/'data/test.csv',dtype={k:v for k,v in V013_DTYPES.items() if k!='pressure'});ids=known.id.to_numpy(copy=True);v037_require(len(ids)==603600 and len(np.unique(ids))==603600,'Approved test IDs invalid')
    sample=pd.read_csv(root/'data/sample_submission.csv',usecols=['id'],dtype=dict(id='int64')).id.to_numpy();v037_require(np.array_equal(ids,sample),'Test/sample row mismatch')
    raw,names,bids=make_sequence_features(known,'combined');del known;gc.collect();v037_require(raw.shape==(7545,80,47),'Approved test feature shape changed')
    checkpoint=v037_path(root,record['checkpoint_path']);path=out/'capacity_laplace_curve_test_predictions.npy';mp=out/'capacity_laplace_curve_test_prediction_metadata.json';identity=dict(checkpoint_sha256=record['checkpoint_sha256'],test_sha256=V011_TEST_SHA,row_ids_sha256=v011_array_sha(ids),sources=sources,arm='v037_all_capacity_laplace_curve')
    if mp.exists():
        meta=json.loads(mp.read_text());v037_require(resume and meta['identity']==identity,'Existing final native-cache identity changed')
        if path.exists():
            p=np.load(path,allow_pickle=False);v037_require(v037_sha(path)==meta['file_sha256'] and v011_array_sha(p)==meta['array_sha256'] and p.dtype==np.float64 and p.shape==ids.shape and np.isfinite(p).all(),'Existing capacity native cache changed');return ids,dict(capacity_laplace_curve=p),dict(capacity_laplace_curve=meta)
    else:meta=None
    predicted=predict_checkpoint_v029(checkpoint,raw,device='cuda',batch_size=512,model_factory=v037_laplace_factory,expected_arm='v037_all_capacity_laplace_curve',expected_loss_kind='laplace_curve',source_fingerprints=sources).astype(np.float64).ravel();v037_require(predicted.shape==ids.shape and np.isfinite(predicted).all(),'Invalid all-data capacity native predictions')
    if path.exists():v037_require(resume and np.array_equal(np.load(path),predicted),'Orphan native prediction array differs')
    else:v011_save_array(path,predicted)
    actual=dict(identity=identity,file=str(path.relative_to(root)),file_sha256=v037_sha(path),array_sha256=v011_array_sha(predicted),row_ids_sha256=v011_array_sha(ids),checkpoint_sha256=record['checkpoint_sha256'],test_sha256=V011_TEST_SHA)
    if meta is not None:v037_require(actual==meta,'Reconstructed missing native cache metadata differs')
    else:v011_write_json(mp,actual)
    return ids,dict(capacity_laplace_curve=predicted),dict(capacity_laplace_curve=actual)

def run_final_v037(root=".",resume=False,expected_F_result_sha256=None,expected_F_audit_sha256=None,expected_source_code_sha256=None):
    root=Path(root).resolve();out=root/"outputs/confirmation_v036/final_all_v037";locks=[];began=time.monotonic();previous_cwd=Path.cwd()
    try:
        os.chdir(root)
        lock=(root/"outputs/.v011_gpu.lock").open('a+');fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(lock)
        v037_require(torch.cuda.is_available(),"Approved actual CUDA required")
        for flag in ["PA3_RUN_V013","PA3_RUN_FINAL_V011"]:v037_require(os.environ.get(flag,"0")!="1","Other notebook run flag enabled")
        current=hashlib.sha256(v037_code(root/V037_NOTEBOOK).encode()).hexdigest();v037_require(current==expected_source_code_sha256,"Final source differs from reviewed pin")
        plan,gate=v037_confirmation_gate(root,expected_F_result_sha256,expected_F_audit_sha256)
        gate["final_source_code_sha256"]=current
        for name,h in V037_SOURCE_DEPENDENCIES.items():
            code=v037_code(root/name);v037_require(hashlib.sha256(code.encode()).hexdigest()==h,"Reusable source changed: "+name);exec(compile(code,name,"exec"),globals())
        old_lock=json.loads((root/"outputs/confirmation_v011/locked_before_D.json").read_text());old_split=json.loads((root/"outputs/confirmation_v011/D_split.json").read_text())
        v011_load_components(root,old_lock["source_fingerprints"]);torch.set_num_threads(4);seed_sequence(20261102)
        laplace_source=v037_code(root/"research/laplace_components_v029.ipynb");v037_require(hashlib.sha256(laplace_source.encode()).hexdigest()==V037_LAPLACE_SHA,"Laplace component changed");exec(compile(laplace_source,"laplace_components_v029.ipynb","exec"),globals())
        v037_require(v037_sha(root/"outputs/confirmation_v014/final_all_v018/submission.csv")==V037_BEST_OFFICIAL_CSV_SHA,"Current official best CSV changed")
        old_path=root/"outputs/confirmation_v011/final_all/final_manifest.json";v037_require(v037_sha(old_path)==V037_ACCEPTED_MANIFEST_SHA,"Accepted five-model manifest changed")
        accepted=json.loads(old_path.read_text());v037_require(v037_sha(v037_path(root,accepted["submission_file"]))==V037_ACCEPTED_CSV_SHA,"Accepted official CSV changed")
        if (out/"final_manifest.json").exists():
            complete=json.loads((out/"final_manifest.json").read_text())
            v037_require(resume and complete["status"]=="complete" and complete["gate"]==gate and complete["weights"]==plan["winner_weights"],"Completed final manifest changed or missing explicit resume")
            for name,h in [("train.csv",V011_TRAIN_SHA),("test.csv",V011_TEST_SHA),("sample_submission.csv",V011_SAMPLE_SHA)]:v037_require(v037_sha(root/"data"/name)==h,"Approved data changed")
            for record in complete["members"].values():v037_require(v037_sha(v037_path(root,record["checkpoint_path"]))==record["checkpoint_sha256"],"Completed final checkpoint changed")
            for path,h in [(v037_path(root,complete["submission"]["file"]),complete["submission_sha256"]),(out/"final_test_predictions.npy",complete["prediction_file_sha256"]),(root/complete["grid_file"],complete["grid_file_sha256"])]:v037_require(v037_sha(path)==h,"Completed final artifact changed")
            return complete
        data=v011_load_all_training(root,old_lock,old_split)
        v037_require(data["data_hashes"]==accepted["all_training_data_fingerprints"] and data["group_sha256"]==accepted["all_training_group_sha256"] and _v8_statistics_sha256(data["prep"])==accepted["all_training_preprocessing_sha256"],"Verified all-training data/statistics changed")
        out.mkdir(parents=True,exist_ok=True)
        sources=dict(old_lock["source_fingerprints"],**V037_SOURCE_DEPENDENCIES,**{V037_NOTEBOOK:current,"confirmed_F_result":expected_F_result_sha256,"confirmed_F_audit":expected_F_audit_sha256,"confirmation_protocol_v036.json":"e9fb1fc5fd7a342fd744f8ed2d7bcdbb4eab3d3e5be8f12da6f93575374ab0f4","laplace_components_v029.ipynb":V037_LAPLACE_SHA})
        design=dict(schema="v037-confirmed-capacity-fixed-postprocessing-final-v1",gate=gate,weights=plan["winner_weights"],new_member_recipe=plan["members"]["capacity_laplace_curve"],all_training_breaths=67905,scored_training_points=2061922,accepted_manifest_sha256=V037_ACCEPTED_MANIFEST_SHA,accepted_CSV_sha256=V037_ACCEPTED_CSV_SHA,source_fingerprints=sources,aggregation="fixed weighted median midpoint then nearest all-training inspiratory grid",unscored_phase_rule="fixed six-member mean",seed=20261102,final_fit_has_independent_holdout=False)
        v011_immutable_json(out/"design.json",design,resume)
        reused={};prior_path=root/V011_PRIOR_DIR/"final_manifest.json";v037_require(v037_sha(prior_path)==V011_PRIOR_MANIFEST_SHA,"Original accepted provenance changed");prior=json.loads(prior_path.read_text());v011_verify_legacy_lock(root,prior)
        for name,record in accepted["members"].items():
            recipe=plan["members"][name];v037_require(record["recipe"]==recipe,"Accepted model recipe no longer matches selected recipe")
            checkpoint=v037_path(root,record["checkpoint_path"]);v037_require(v037_sha(checkpoint)==record["checkpoint_sha256"],"Accepted checkpoint changed")
            if name in ["deep","original_compact"]:
                verified=v011_reuse_original(root,dict(member_id=name,recipe=recipe),data,prior)
            else:
                rp=v037_path(root,record["report_file"]);v037_require(v037_sha(rp)==record["report_sha256"],"Accepted report changed")
                report=json.loads(rp.read_text());v037_require(report["completed"] and report["training_breaths"]==67905 and not report["validation_used"] and report["executed_epochs"]==recipe["selected_epochs"] and report["executed_lr_schedule"]==recipe["lr_schedule"],"Accepted all-training budget incomplete")
                model,prep,payload=load_checkpoint_v008(checkpoint,device="cpu",model_factory=v011_factory(name),expected_arm=record["arm"],expected_loss_kind=recipe["loss_kind"],source_fingerprints=accepted["source_fingerprints"])
                v011_check_fixed_payload(payload,dict(recipe=recipe),data["data_hashes"],data["prep"]);del model,prep,payload
                verified=record
            reused[name]=dict(verified,reuse_controller="v037",accepted_manifest_sha256=V037_ACCEPTED_MANIFEST_SHA);gc.collect()
        # Check every reused inference artifact before any new optimizer update.
        test_ids=pd.read_csv(root/"data/test.csv",usecols=["id"],dtype=dict(id="int64")).id.to_numpy()
        sample_ids=pd.read_csv(root/"data/sample_submission.csv",usecols=["id"],dtype=dict(id="int64")).id.to_numpy()
        v037_require(len(test_ids)==603600 and len(np.unique(test_ids))==603600 and np.array_equal(test_ids,sample_ids),"Approved test/sample IDs changed")
        v037_require(v037_sha(v037_path(root,accepted["test_row_ids_file"]))==accepted["test_row_ids_file_sha256"] and np.array_equal(test_ids,np.load(v037_path(root,accepted["test_row_ids_file"]))),"Accepted test rows changed")
        for name,record in reused.items():
            meta=accepted["prediction_members"][name];path=v037_path(root,meta["file"]);v037_require(v037_sha(path)==meta["file_sha256"],"Accepted inference cache changed before training")
            a=np.load(path,allow_pickle=False)
            v037_require(a.dtype==np.float64 and a.shape==(603600,) and np.isfinite(a).all() and v011_array_sha(a)==meta["array_sha256"] and meta["row_ids_sha256"]==v011_array_sha(test_ids) and meta["checkpoint_sha256"]==record["checkpoint_sha256"] and meta["test_sha256"]==V011_TEST_SHA,"Accepted inference cache identity changed before training")
        v037_require(all(len(h)==64 and all(c in "0123456789abcdef" for c in h) for h in sources.values()),"Source fingerprint is not a SHA256 digest")
        grid=pressure_codebook_v016(data["y"],data["out"])
        new_record=v037_fit_capacity(root,out,copy.deepcopy(plan['members']['capacity_laplace_curve']),data,gate,sources,resume)
        training_meta=dict(group_sha256=data['group_sha256'],data_fingerprints=data['data_hashes'],preprocessing_sha256=_v8_statistics_sha256(data['prep']))
        del data;gc.collect();torch.cuda.empty_cache()
        ids,new_predictions,new_prediction_meta=v037_infer_capacity(root,out,new_record,sources,resume)
        predictions=dict(new_predictions);prediction_meta=dict(new_prediction_meta)
        for name,record in reused.items():
            meta=accepted["prediction_members"][name];p=v037_path(root,meta["file"]);v037_require(v037_sha(p)==meta["file_sha256"],"Accepted cached test predictions changed")
            a=np.load(p,allow_pickle=False)
            v037_require(a.dtype==np.float64 and a.shape==ids.shape and np.isfinite(a).all() and v011_array_sha(a)==meta["array_sha256"] and meta["row_ids_sha256"]==v011_array_sha(ids) and meta["checkpoint_sha256"]==record["checkpoint_sha256"] and meta["test_sha256"]==V011_TEST_SHA,"Accepted test cache identity changed")
            predictions[name]=a;prediction_meta[name]=meta
        weights=plan["winner_weights"];base=sum(w*predictions[k] for k,w in weights.items());processed=selected_postprocessing_v016(predictions,weights,grid)
        phase=pd.read_csv(root/"data/test.csv",usecols=["id","u_out"],dtype=dict(id="int64",u_out="int8"))
        v037_require(np.array_equal(phase.id.to_numpy(),ids),"Test phase/ID alignment changed")
        final=np.where(phase.u_out.to_numpy()==0,processed,base);v037_require(np.isfinite(final).all() and final.shape==(603600,),"Final predictions invalid")
        for path,value in [(out/"all_training_inspiratory_pressure_grid.npy",grid),(out/"test_row_ids.npy",ids),(out/"final_test_predictions.npy",final)]:
            if path.exists():v037_require(resume and np.array_equal(np.load(path),value),"Existing final array changed")
            else:v011_save_array(path,value)
        submission=v011_write_submission(out,ids,final,resume)
        v037_require(v037_sha(v037_path(root,accepted["submission_file"]))==V037_ACCEPTED_CSV_SHA,"Best official CSV changed during work")
        _,after=v037_confirmation_gate(root,expected_F_result_sha256,expected_F_audit_sha256);v037_require(dict(after,final_source_code_sha256=current)==gate,"Final prerequisite changed during work")
        v037_require(v037_sha(root/"outputs/confirmation_v014/final_all_v018/submission.csv")==V037_BEST_OFFICIAL_CSV_SHA,"Official0.17427 CSV changed during final work")
        manifest=dict(status="complete",schema=design["schema"],gate=gate,design_sha256=v011_json_sha(design),members=dict(reused,capacity_laplace_curve=new_record),prediction_members=prediction_meta,weights=weights,aggregation=design["aggregation"],unscored_phase_rule=design["unscored_phase_rule"],source_fingerprints=sources,training=training_meta,all_training_breaths=67905,scored_training_points=2061922,grid_file=str((out/"all_training_inspiratory_pressure_grid.npy").relative_to(root)),grid_file_sha256=v037_sha(out/"all_training_inspiratory_pressure_grid.npy"),grid_levels=len(grid),prediction_file_sha256=v037_sha(out/"final_test_predictions.npy"),prediction_array_sha256=v011_array_sha(final),row_ids_array_sha256=v011_array_sha(ids),submission=submission,submission_sha256=submission["file_sha256"],old_CSV_sha256=V037_ACCEPTED_CSV_SHA,final_fit_has_independent_holdout=False,elapsed_seconds=time.monotonic()-began,submitted=False,completed_utc=datetime.now(timezone.utc).isoformat())
        v011_write_json(out/"final_manifest.json",manifest);return manifest
    finally:
        os.chdir(previous_cwd)
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get("PA3_RUN_FINAL_V037","0")=="1":
    V037_RESULT=run_final_v037(Path.cwd(),resume=os.environ.get("PA3_RESUME_FINAL_V037","0")=="1",expected_F_result_sha256=os.environ.get("PA3_V037_F_RESULT_SHA256"),expected_F_audit_sha256=os.environ.get("PA3_V037_F_AUDIT_SHA256"),expected_source_code_sha256=os.environ.get("PA3_V037_SOURCE_SHA256"))
    print(json.dumps(V037_RESULT["submission"],indent=2))
